In [1]:
import os
print(os.path.exists('.env'))
print(os.getcwd())

True
/Users/anugyasahu/Desktop/Langchain_HandsON


#### Langchain LLM Model using API

In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
answer =llm.invoke("What is the capital of Germany?")
answer.content


'The capital of Germany is Berlin.'

#### Prompt Templates 
helps to translate user input and parameters into instructions for a language model, it generates coherent outputs

Input - dictionary , each key represents a variable in the prompt template to fill in

A. String Prompt Templates - used to format a single string , and generally are used for simpler inputs

B. Chat Prompt Templates - used to format a list of messages, can define system prompt also - for e.g. ACt as an insurance expert, acts as a helpful assistant (system prompts)

C. Messages Placeholder - adding a list of messages in a particular place, dont need to pass system and user prompt in strings. This is used when we want the user to pass in a list of messages and we would slot into particular spot.



A. String Prompt Template

In [1]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template("What is the capital of {country}?")
prompt = prompt_template.format(country="Germany")

In [2]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
response = llm.invoke(prompt)
print(response.content)

The capital of Germany is Berlin.


B. Chat Prompt Template

In [5]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "What is the capital of {country}?")   
])

prompt = chat_template.invoke({"country": "Germany"})
response = llm.invoke(prompt)
print(response.content)

The capital of Germany is Berlin.


C. Message PlaceHolder


In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("msgs")
    ])
# can also give the messages directly to the prompt template
prompt = prompt_template.invoke({"msgs": [HumanMessage(content="What is the capital of Germany?")]})

llm.invoke(prompt).content

'The capital of Germany is Berlin.'

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("msgs")
    ])

# Define multiple messages
messages = [
    HumanMessage(content="What is the capital of Germany?"),
    AIMessage(content="The capital of Germany is Berlin."),
    HumanMessage(content="What is the population of Berlin?")
]

# Invoke the prompt template with the messages

output = prompt_template.invoke({"msgs": messages})
print(output)

messages=[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Germany?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Germany is Berlin.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the population of Berlin?', additional_kwargs={}, response_metadata={})]


In [11]:
llm.invoke(output).content

"As of my last update in October 2023, the population of Berlin is approximately 3.7 million people. However, population figures can change, so it's a good idea to check the latest statistics for the most current information."

#### Structured Outputs

Scenarios where we need models to output in a strcutured format, desired - storing model output in a database and ensure that the output conforms to database schema, needs the LLM to support a special feature called "function calling"

Functions to use - 

with_structured_output ----- ChatGPT (llm.with_structured_output.invoke())
-- always define output schema
output parser ----- HuggingFace OpenSource models

Schemas - Types 

A. TypedDict - define datatype, cant validate data, faster 

B. Pydantic - data validation at runtime, define default values and descriptions

C. JSON - json api desired, hard to define datatypes

A. TypedDict

In [ ]:
from typing import List, Optional, Annotated, TypedDict

class MovieInfo(TypedDict):
    title: Annotated[str, "The title of the movie"]
    director: Annotated[str, "The director of the movie"]
    release_year: Annotated[int, "The year the movie was released"]
    genres: Annotated[List[str], "The genres of the movie"]
    rating: Annotated[Optional[float], "The rating of the movie"]
    box_office: Annotated[Optional[float], "The box office earnings of the movie in USD"]

structured_llm = llm.with_structured_output(MovieInfo)

result = structured_llm.invoke("Provide information about the movie Shutter Island.")

# print(result)
# print(type(result))

# problem, as title is defined as str but we are passing an int, it will just print the int value without any error, which is not expected
# cant validate the data type of the output, which is a major drawback of using TypedDict for structured output
movie = MovieInfo({"title":12})
print(movie["title"])

12


B. Pydantic

In [14]:
from pydantic import BaseModel, Field
class MovieInfoModel(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    genres: List[str] = Field(..., description="The genres of the movie")
    rating: Optional[float] = Field(None, description="The rating of the movie")
    box_office: Optional[float] = Field(None, description="The box office earnings of the movie in USD")

structured_llm = llm.with_structured_output(MovieInfoModel)
result = structured_llm.invoke("Provide information about the movie Shutter Island.")
print(result)
print(type(result))

print(result.model_dump()) # to convert the pydantic model to a dictionary
print(result.model_dump_json()) # to convert the pydantic model to a json string

title='Shutter Island' director='Martin Scorsese' release_year=2010 genres=['Mystery', 'Thriller', 'Psychological'] rating=8.2 box_office=294804195.0
<class '__main__.MovieInfoModel'>
{'title': 'Shutter Island', 'director': 'Martin Scorsese', 'release_year': 2010, 'genres': ['Mystery', 'Thriller', 'Psychological'], 'rating': 8.2, 'box_office': 294804195.0}
{"title":"Shutter Island","director":"Martin Scorsese","release_year":2010,"genres":["Mystery","Thriller","Psychological"],"rating":8.2,"box_office":294804195.0}


In [16]:
# now if we try to create an instance of the MovieInfoModel with an incorrect data type, it will raise a validation error, which is expected

try:
    movie = MovieInfoModel(title=12, director="Martin Scorsese", release_year=2010, genres=["Thriller", "Mystery"], rating=8.2, box_office=294.8)
    print(movie)
except Exception as e:
    print(f"Error occurred: {e}")

Error occurred: 1 validation error for MovieInfoModel
title
  Input should be a valid string [type=string_type, input_value=12, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


C. JSON Schema

In [23]:
# json schema for Langchain /OpenAI function calling

movie_json_schema = {"name": "MovieInfo", "description": "A schema representing information about a movie", "type": "object", "properties": {"title": {"type": "string", "description": "The title of the movie"}, "director": {"type": "string", "description": "The director of the movie"}, "release_year": {"type": "integer", "description": "The year the movie was released"}, "genres": {"type": "array", "items": {"type": "string"}, "description": "The genres of the movie"}, "rating": {"type": ["number", "null"], "description": "The rating of the movie"}, "box_office": {"type": ["number", "null"], "description": "The box office earnings of the movie in USD"}}}

from langchain_core.utils.function_calling import convert_to_openai_function

structured_llm = llm.with_structured_output(movie_json_schema, method="json_mode")
result = structured_llm.invoke("Provide information about the movie Shutter Island. Return as JSON.")
print(result)

{'title': 'Shutter Island', 'release_year': 2010, 'director': 'Martin Scorsese', 'screenplay': 'Laeta Kalogridis', 'based_on': 'Shutter Island by Dennis Lehane', 'genre': ['Mystery', 'Thriller', 'Psychological'], 'cast': [{'name': 'Leonardo DiCaprio', 'character': 'Teddy Daniels'}, {'name': 'Mark Ruffalo', 'character': 'Chuck Aule'}, {'name': 'Ben Kingsley', 'character': 'Dr. John Cawley'}, {'name': 'Michelle Williams', 'character': 'Dolores Chanal'}, {'name': 'Max von Sydow', 'character': 'Dr. Naehring'}], 'plot_summary': 'In 1954, U.S. Marshal Teddy Daniels and his new partner Chuck Aule are sent to a remote mental institution on Shutter Island to investigate the disappearance of a patient named Rachel Solando. As they delve deeper into the case, they uncover shocking truths about the institution and their own pasts.', 'themes': ['Mental Illness', 'Reality vs. Illusion', 'Guilt and Trauma', 'Justice and Injustice'], 'runtime': '138 minutes', 'budget': '$80 million', 'box_office': '$2

#### Output Parser

takes the output of a model and transforms it to a more suitable format for downstream tasks. Useful when using LLMs for structured data or to normalize output from chat models and LLMs (any LLMs)

Types - String, csv, json, OutputFixing, Pydantic, YAML, XML etc

 JsonOutputParser - tells the LLM to return valid JSON and then in a Python object 
 - includes .get_format_instructions() (how to format the output)
 - no field/type validation - structure may differ if the LLM deviates
  
PydanticOutputParser - enforces the LLM's output matches a strict schema defined by a pydantic model
- schema validation
- automaically generates format instructions for LLM
- must handle errors as if no match, parsing fails

# String output parser

In [ ]:

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

template = PromptTemplate.from_template("What is the capital of {country}?")
chat = template.invoke({"country": "France"})
result = llm.invoke(chat)
print(result.content)

The capital of France is Paris.


Advantages -

1. When using chain
2. Dont have to print .content

In [4]:
# Using Parser

parser = StrOutputParser()

template = PromptTemplate.from_template("What is the capital of {country}?")
chat = template.invoke({"country": "France"})
result = llm.invoke(chat)

parser_result = parser.invoke(result)

print(parser_result)

The capital of France is Paris.


In [5]:
# With chain
parser = StrOutputParser()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
template = PromptTemplate.from_template("What is the capital of {country}?")

chain = template | llm | parser
result =chain.invoke({"country": "India"})

print(result)


The capital of India is New Delhi.


# Json Output Parser

In [7]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

template = PromptTemplate.from_template("""
    Extract the following product information from the text and return it in JSON format:
- Product Name
- Price
- Category
- features (list)

Text: {text}
format_instructions: {format_instructions}
""",

partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

chain = template | llm | json_parser
response = chain.invoke({"text": "The iPhone 12 is a great smartphone with a 6.1-inch OLED display and a powerful A14 Bionic chip."})
print(response)

{'Product Name': 'iPhone 12', 'Price': None, 'Category': 'smartphone', 'features': ['6.1-inch OLED display', 'A14 Bionic chip']}


# Pydantic Output Parser

In [8]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

class ProductInfo(BaseModel):
    product_name: str = Field(..., description="The name of the product")
    price: str = Field(..., description="The price of the product")
    category: str = Field(..., description="The category of the product")
    features: list[str] = Field(..., description="The features of the product")

pydantic_parser = PydanticOutputParser(pydantic_object=ProductInfo)

template = PromptTemplate.from_template("""
                                        Extract the following product information from the text: {text}
                                        format_instructions: {format_instructions}
                                        """,
                                        partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
                                        )

chain = template | llm | pydantic_parser

text = """The Samsung Galaxy S21 is a flagship smartphone with a 6.2-inch AMOLED display, Exynos 2100 processor, and a triple camera setup."""

result = chain.invoke({"text": text})
print(result)

product_name='Samsung Galaxy S21' price='not specified' category='smartphone' features=['6.2-inch AMOLED display', 'Exynos 2100 processor', 'triple camera setup']


# Chains